In [1]:
# In this file we are doing cross validation and model selection
# cross validation refers to splitting the training data in n time and train model

# Cross-validation is a technique where:
# You split data multiple times
# Train and test the model multiple times
# Then average the results

In [50]:
import pandas as pd 
import numpy as np 
from sklearn.model_selection import StratifiedShuffleSplit, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, TargetEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import root_mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

In [29]:
df = pd.read_csv('housing.csv')

In [30]:
df

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY
...,...,...,...,...,...,...,...,...,...,...
20635,-121.09,39.48,25.0,1665.0,374.0,845.0,330.0,1.5603,78100.0,INLAND
20636,-121.21,39.49,18.0,697.0,150.0,356.0,114.0,2.5568,77100.0,INLAND
20637,-121.22,39.43,17.0,2254.0,485.0,1007.0,433.0,1.7000,92300.0,INLAND
20638,-121.32,39.43,18.0,1860.0,409.0,741.0,349.0,1.8672,84700.0,INLAND


In [31]:
df['income_cat'] = pd.cut(df['median_income'], bins=[0, 1.5, 3.0, 4.5, 6.0, np.inf], labels=[1,2,3,4,5])

strat = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_idx, test_idx in strat.split(df, df['income_cat']):
    x_train = df.loc[train_idx].drop('income_cat', axis=1)
    x_test = df.loc[test_idx].drop('income_cat', axis=1)

training = x_train.copy() 

x = training.drop('median_house_value', axis=1)
y = training['median_house_value']

In [42]:
x_num = x.drop('ocean_proximity', axis=1).columns.to_list()
x_cat = ['ocean_proximity']

In [43]:
cat_pipe = Pipeline([
    ('hotencode', OneHotEncoder(handle_unknown='ignore'))
])
num_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')), 
    ('stand', StandardScaler())
])

final = ColumnTransformer([ 
    ('num', num_pipe, x_num),
    ('cat', cat_pipe, x_cat)
])

In [48]:
xt = final.fit_transform(x)

In [66]:
# linear regression 
lin = LinearRegression()
lin_model = lin.fit(xt,y)
lpred = lin_model.predict(xt)

rmse = root_mean_squared_error(y, lpred)
print("RMSE : ", rmse)
print("score : ",lin_model.score(xt,y))

# cross validation

# WARNING: Scikit-Learn’s scoring uses utility functions (higher is better), so RMSE is returned as negative.
# We use minus (-) to convert it back to positive RMSE.

cv = -cross_val_score(lin_model, xt, y, scoring="neg_root_mean_squared_error", cv=10)

print("CV rmse : ", cv) 
print("describe :\n", pd.Series(cv).describe())

RMSE :  69050.56219504568
score :  0.6438078994746375
CV rmse :  [72229.03469752 65318.2240289  67706.39604745 69368.53738998
 66767.61061621 73003.75273869 70522.24414582 69440.77896541
 66930.32945876 70756.31946074]
describe :
 count       10.000000
mean     69204.322755
std       2500.382157
min      65318.224029
25%      67124.346106
50%      69404.658178
75%      70697.800632
max      73003.752739
dtype: float64


In [72]:
# DecisionTreeRegressor
# in decision tree there is always a chance of overfitting 
des = DecisionTreeRegressor()
d_model = des.fit(xt,y) 
dpred = d_model.predict(xt)

drmse = root_mean_squared_error(y, dpred)
print("RMSE : ",drmse)
print("score",d_model.score(xt, y))


# cross validation

cv = -cross_val_score(d_model, xt, y, scoring="neg_root_mean_squared_error", cv=10)
print("CV rmse : ", dcv) 
print("describe :\n", pd.Series(dcv).describe())

RMSE :  0.0
score 1.0
CV rmse :  [69381.15237496 69081.29428621 64899.63133816 68575.44027215
 67774.16674195 69067.62811174 71797.60398858 69755.96581543
 68518.77008847 71773.85763812]
describe :
 count       10.000000
mean     69062.551066
std       1970.257039
min      64899.631338
25%      68532.937634
50%      69074.461199
75%      69662.262455
max      71797.603989
dtype: float64


In [73]:
# RandomforestRegressor
ran = RandomForestRegressor()
r_model = ran.fit(xt, y) 
rpred = r_model.predict(xt) 

rrmse = root_mean_squared_error(y, rpred)
print("RMSE : ", rrmse)
print("score : ",r_model.score(xt, y))

# cross validation
cv = -cross_val_score(r_model, xt, y, scoring="neg_root_mean_squared_error", cv=10)
print("CV rmse : ", cv) 
print("describe :\n", pd.Series(cv).describe())

RMSE :  18409.35469348552
score :  0.9746821410129108
CV rmse :  [50810.77816537 49451.2644542  46408.98072041 50469.88539684
 47266.62043072 49456.10719184 51675.40556386 49158.61397456
 47439.64038784 53116.86934114]
describe :
 count       10.000000
mean     49525.416563
std       2091.644824
min      46408.980720
25%      47869.383785
50%      49453.685823
75%      50725.554973
max      53116.869341
dtype: float64
